# Allstate Claims Severity


<img src='https://www.adasigorta.com.tr/trex/assets/img/blog/2762329745.jpg'>


Bu çalışmada `Allstate Claims Severity` yarışması için sigorta müşteri ve talep özellikleri kullanılarak hasar maliyeti tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/allstate-claims-severity.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.csv', 'train.csv.zip'])
test_member = find_zip_member(zip_members, ['test.csv', 'test.csv.zip'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [6]:
def read_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f)
            return pd.read_csv(f)

train = read_csv_from_zip(zip_path, train_member)
test = read_csv_from_zip(zip_path, test_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (188318, 132)
Test shape: (125546, 131)
Sample submission shape: (125546, 2)


,id,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,cat10,cat11,cat12,cat13,cat14,cat15,cat16,cat17,cat18,cat19,cat20,cat21,cat22,cat23,cat24,cat25,cat26,cat27,cat28,cat29,cat30,cat31,cat32,cat33,cat34,cat35,cat36,cat37,cat38,cat39,cat40,cat41,cat42,cat43,cat44,cat45,cat46,cat47,cat48,cat49,cat50,cat51,cat52,cat53,cat54,cat55,cat56,cat57,cat58,cat59,cat60,cat61,cat62,cat63,cat64,cat65,cat66,cat67,cat68,cat69,cat70,cat71,cat72,cat73,cat74,cat75,cat76,cat77,cat78,cat79,cat80,cat81,cat82,cat83,cat84,cat85,cat86,cat87,cat88,cat89,cat90,cat91,cat92,cat93,cat94,cat95,cat96,cat97,cat98,cat99,cat100,cat101,cat102,cat103,cat104,cat105,cat106,cat107,cat108,cat109,cat110,cat111,cat112,cat113,cat114,cat115,cat116,cont1,cont2,cont3,cont4,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,loss
0,1,A,B,A,B,A,A,A,A,B,A,B,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,D,B,B,D,D,B,D,C,B,D,B,A,A,A,A,A,D,B,C,E,A,C,T,B,G,A,A,I,E,G,J,G,BU,BC,C,AS,S,A,O,LB,0.726300,0.245921,0.187583,0.789639,0.310061,0.718367,0.335060,0.30260,0.67135,0.83510,0.569745,0.594646,0.822493,0.714843,2213.18
1,2,A,B,A,A,A,A,A,A,B,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,D,B,B,D,D,A,B,C,B,D,B,A,A,A,A,A,D,D,C,E,E,D,T,L,F,A,A,E,E,I,K,K,BI,CQ,A,AV,BM,A,O,DP,0.330514,0.737068,0.592681,0.614134,0.885834,0.438917,0.436585,0.60087,0.35127,0.43919,0.338312,0.366307,0.611431,0.304496,1283.60
2,5,A,B,A,A,B,A,A,A,B,B,B,B,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,D,B,B,B,D,B,D,C,B,B,B,A,A,A,A,A,D,D,C,E,E,A,D,L,O,A,B,E,F,H,F,A,AB,DK,A,C,AF,A,I,GK,0.261841,0.358319,0.484196,0.236924,0.397069,0.289648,0.315545,0.27320,0.26076,0.32446,0.381398,0.373424,0.195709,0.774425,3005.09
3,10,B,B,A,B,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,D,B,B,D,D,D,B,C,B,D,B,A,A,A,A,A,D,D,C,E,E,D,T,I,D,A,A,E,E,I,K,K,BI,CS,C,N,AE,A,O,DJ,0.321594,0.555782,0.527991,0.373816,0.422268,0.440945,0.391128,0.31796,0.32128,0.44467,0.327915,0.321570,0.605077,0.602642,939.85
4,11,A,B,A,B,A,A,A,A,B,B,A,B,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,D,B,D,B,D,B,B,C,B,B,C,A,A,A,B,H,D,B,D,E,E,A,P,F,J,A,A,D,E,K,G,B,H,C,C,Y,BM,A,K,CK,0.273204,0.159990,0.527991,0.473202,0.704268,0.178193,0.247408,0.24564,0.22089,0.21230,0.204687,0.202213,0.246011,0.432606,2763.85


## 4. Ön İşleme


In [7]:
# Bu bölümde hedef değişkeni ayırıyor ve veri tiplerini hazırlıyoruz.


In [8]:
target_col = 'loss'
feature_cols = [col for col in train.columns if col not in ['id', target_col] and col in test.columns]

train_x = train[feature_cols].copy()
train_y = np.log1p(train[target_col])
test_x = test[feature_cols].copy()


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde kategorik ve sayısal özellikleri modele uygun hale getiriyoruz.


In [10]:
categorical_cols = [col for col in feature_cols if train_x[col].dtype == 'object']
for col in categorical_cols:
    train_x[col] = train_x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

numeric_cols = [col for col in feature_cols if col not in categorical_cols]
for col in numeric_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (150654, 130)
x_valid shape: (37664, 130)


## 6. Model Kurma


In [11]:
# Bu bölümde hasar maliyeti tahmini için CatBoost modelini kuruyoruz.


In [12]:
model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=300, learning_rate=0.08, loss_function='RMSE', random_seed=42, verbose=0)

## 7. RMSE Değerlendirmesi


In [13]:
# Bu bölümde validation verisi üzerindeki RMSE değerini hesaplıyoruz.


In [14]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(np.expm1(y_valid), np.expm1(valid_preds))
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 1910.89904


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [16]:
test_preds = model.predict(test_x)
submission = sample_submission.copy()
if 'loss' in submission.columns:
    submission['loss'] = np.expm1(test_preds[:len(submission)])
else:
    submission.iloc[:, 1] = np.expm1(test_preds[:len(submission)])

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (125546, 2)


,id,loss
0,4,1522.084116
1,6,1918.185470
2,9,8464.299378
3,12,5631.043060
4,15,847.458967


In [17]:
output_path = '/content/allstate_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/allstate_submission.csv


## 9. Sonuç


Bu çalışmada Allstate Claims Severity yarışması için sigorta müşteri ve talep özellikleri kullanılarak hasar maliyeti tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 1910.89904 RMSE değeri elde etti ve test verisi için hasar maliyeti tahminleri üretildi.
